# **Step 1: Download libraries**

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

## Step 2: Download dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kumarajarshi/life-expectancy-who")

print("Path to dataset files:", path)

In [ ]:
import os
#construct file path (downloaded)
data_file=os.path.join(path,"Life Expectancy Data.csv")
#load csv file into dataframe
df= pd.read_csv(data_file)
#display top 5 rows of loaded dataset
df.head()


In [ ]:
df.tail()

# **Step 3: Sanity Check**

In [ ]:
#Shape of df
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
#find missing values
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
#displaying selected rows with some condition

nepal_data=df[df['Country']== 'Nepal']
print(nepal_data)

# **Step 4: Exploratory Data Analysis (EDA)**

In [ ]:
dfc=df.copy()
dfc.head()

In [ ]:
dfc.describe()

In [ ]:
#To understand distrubution, check histogram
for i in dfc.select_dtypes(include="number"):
  sns.histplot(data=dfc, x=i)
  plt.show()

In [ ]:
dfco=dfc['Year']
# dfco.head()

sns.histplot(data=dfc,x=dfco)

In [ ]:
#outliers observation
#To understand distrubution, check histogram
for i in dfc.select_dtypes(include="number"):
  sns.boxplot(data=dfc, x=i)
  plt.show()

In [ ]:
dfc.select_dtypes(include="number").columns

In [ ]:
dfc.columns=dfc.columns.str.strip()

In [ ]:
dfc.select_dtypes(include="number").columns

In [ ]:
#scatterplot to check corelation between target variable to respective numeric column

for i in ['Year', 'Adult Mortality', 'infant deaths',
       'Alcohol', 'percentage expenditure', 'Hepatitis B', 'Measles', 'BMI',
       'under-five deaths', 'Polio', 'Total expenditure', 'Diphtheria',
       'HIV/AIDS', 'GDP', 'Population', 'thinness  1-19 years',
       'thinness 5-9 years', 'Income composition of resources', 'Schooling']:
       sns.scatterplot(data=dfc, x=i, y='Life expectancy')
       plt.show()


In [ ]:
#correlation with heatmap
selected_data=dfc.select_dtypes(include="number").corr()
plt.figure(figsize=(15,15))
sns.heatmap(selected_data,annot=True)

# **Step 5: Missing Value Treatment**

In [ ]:
dfc.isnull().sum()

In [ ]:
dfc.info()

## For Numeric Columns

| Strategy | When to Use |
|---|---|
| Mean | Data is normally distributed |
| Median | Data has outliers |
| Mode | Repeated values |
| Interpolation | Time series |

In [ ]:
for i in ["BMI", "Polio", "Income composition of resources"]:
  dfc[i].fillna(dfc[i].mean(), inplace=True)

In [ ]:
dfc.isnull().sum()

In [ ]:
#using sklearn to treat missing values
from sklearn.impute import KNNImputer
imputer = KNNImputer()

In [ ]:
for i in dfc.select_dtypes(include="number").columns:
  dfc[i]=imputer.fit_transform(dfc[[i]])

In [ ]:
dfc.isnull().sum()

# **Ranking feature vectores based on corr**

In [ ]:
# Calculate correlation matrix
corr_matrix = dfc.corr(numeric_only=True)

# Correlation with target variable
target_corr = corr_matrix['Life expectancy']

# Sort correlations
ranked_corr = target_corr.sort_values(ascending=False)

print(ranked_corr)

In [ ]:
dfc_subset = dfc[["Schooling", "Life expectancy"]]

dfc_subset.head()

# **Train test split**

In [ ]:
from sklearn.model_selection import train_test_split

# Features (input variables)
X = dfc_subset.drop(columns=["Life expectancy"])

# Target variable
y = dfc_subset["Life expectancy"]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
import numpy as np

# Convert training data to numpy arrays
x = X_train.values.flatten()
y = y_train.values.flatten()

# Number of observations
n = len(x)

# Required summations
sum_x = np.sum(x)
sum_y = np.sum(y)
sum_xy = np.sum(x * y)
sum_x2 = np.sum(x ** 2)

# Calculate b1 (slope)
b1 = (n * sum_xy - sum_x * sum_y) / (n * sum_x2 - (sum_x ** 2))

# Calculate b0 (intercept)
b0 = (sum_y * sum_x2 - sum_x * sum_xy) / (n * sum_x2 - (sum_x ** 2))

# Print coefficients
print("Slope (b1):", b1)
print("Intercept (b0):", b0)

# Regression equation
print(f"Regression Equation: y = {b0:.2f} + {b1:.2f}x")


In [ ]:
import pandas as pd
# MAE function
def MAE(y_actual, y_predicted):
    return np.mean(np.abs(y_actual - y_predicted))

# Convert X_test to 1D array
x_test = X_test.values.flatten()

# Predicted values for all test values x_test
Yp = b0 + b1 * x_test

# Create comparison table
results = pd.DataFrame({
    "Actual_Y": y_test.values,
    "Predicted_Y": Yp
})

# Calculate error
results["Error"] = results["Actual_Y"] - results["Predicted_Y"]

# Display first rows
print(results.head())

# Calculate error
results_mae["Error"] = MEA(results["Actual_Y"], results["Predicted_Y"])

# Display first rows
print(results_mae.head())

# Regression Error Metrics

| Metric | Full Form | Formula | Purpose |
|---|---|---|---|
| MAE | Mean Absolute Error | MAE = (1/n) Σ|yi - ŷi| | Average absolute error |
| MSE | Mean Squared Error | MSE = (1/n) Σ(yi - ŷi)² | Penalizes large errors |
| RMSE | Root Mean Squared Error | RMSE = √MSE | Error in original unit |
| SSE | Sum of Squared Errors | SSE = Σ(yi - ŷi)² | Total squared error |
| SAE | Sum of Absolute Errors | SAE = Σ|yi - ŷi| | Total absolute error |
| RSE | Residual Standard Error | RSE = √(SSE / (n-p-1)) | Standard deviation of residuals |
| R² | R-Squared | 1 - (SSE/SST) | Goodness of fit |
| Adjusted R² | Adjusted R-Squared | Adjusts R² for number of features | Better for multiple regression |
| MAPE | Mean Absolute Percentage Error | (100/n) Σ|(yi-ŷi)/yi| | Percentage error |
| MSLE | Mean Squared Log Error | (1/n) Σ(log(1+yi)-log(1+ŷi))² | Useful for exponential growth data |
| RMSLE | Root Mean Squared Log Error | √MSLE | Log-scaled RMSE |
| Explained Variance | Explained Variance Score | Measures explained variability | Model quality |


In [ ]:
import pandas as pd
import numpy as np

# MAE function
def MAE(y_actual, y_predicted):
    return np.mean(np.abs(y_actual - y_predicted))

# Convert X_test to 1D array
x_test = X_test.values.flatten()

# Predicted values
Yp = b0 + b1 * x_test

# Create comparison table
results = pd.DataFrame({
    "Actual_Y": y_test.values,
    "Predicted_Y": Yp
})

# Row-wise error
results["Error"] = results["Actual_Y"] - results["Predicted_Y"]

# Display first rows
print(results)

# Calculate MAE
mae_value = MAE(results["Actual_Y"], results["Predicted_Y"])

print("Mean Absolute Error (MAE):", mae_value)